# 03 - U-Net Model Implementation

## Objective
Build a U-Net architecture using PyTorch for microscopy image denoising.

## Architecture
1. Encoder
2. Bottleneck
3. Decoder
4. Skip Connections
5. Complete U-Net Model

## Input and Output
- Input: Noisy microscopy image
- Target: Clean ground-truth microscopy image
- Output: Denoised microscopy image

In [1]:
import torch
import torch.nn as nn

# Create a sample batch
x = torch.randn(8, 1, 256, 256)

print("Input shape:", x.shape)

# Define convolution layer
conv = nn.Conv2d(
    in_channels=1,
    out_channels=16,
    kernel_size=3,
    stride=1,
    padding=1
)

# Apply convolution
output = conv(x)

print("Output shape:", output.shape)

Input shape: torch.Size([8, 1, 256, 256])
Output shape: torch.Size([8, 16, 256, 256])


In [2]:
# Define ReLU activation
relu = nn.ReLU()

# Apply ReLU to convolution output
activated_output = relu(output)

print("Before ReLU:")
print("Minimum:", output.min().item())
print("Maximum:", output.max().item())

print("\nAfter ReLU:")
print("Minimum:", activated_output.min().item())
print("Maximum:", activated_output.max().item())

print("Output shape:", activated_output.shape)

Before ReLU:
Minimum: -3.807046890258789
Maximum: 3.890408754348755

After ReLU:
Minimum: 0.0
Maximum: 3.890408754348755
Output shape: torch.Size([8, 16, 256, 256])


In [3]:
# Define Max Pooling
pool = nn.MaxPool2d(
    kernel_size=2,
    stride=2
)

# Apply pooling after ReLU
pooled_output = pool(activated_output)

print("Before pooling:", activated_output.shape)
print("After pooling:", pooled_output.shape)

Before pooling: torch.Size([8, 16, 256, 256])
After pooling: torch.Size([8, 16, 128, 128])


In [4]:
# Define upsampling
upsample = nn.Upsample(
    scale_factor=2,
    mode="bilinear",
    align_corners=False
)

# Apply upsampling
upsampled_output = upsample(pooled_output)

print("Before upsampling:", pooled_output.shape)
print("After upsampling:", upsampled_output.shape)

Before upsampling: torch.Size([8, 16, 128, 128])
After upsampling: torch.Size([8, 16, 256, 256])


In [5]:
# Simulate encoder and decoder feature maps

encoder_features = torch.randn(8, 16, 256, 256)

decoder_features = torch.randn(8, 16, 256, 256)

# Concatenate along channel dimension
combined_features = torch.cat(
    [encoder_features, decoder_features],
    dim=1
)

print("Encoder shape:", encoder_features.shape)
print("Decoder shape:", decoder_features.shape)
print("Combined shape:", combined_features.shape)

Encoder shape: torch.Size([8, 16, 256, 256])
Decoder shape: torch.Size([8, 16, 256, 256])
Combined shape: torch.Size([8, 32, 256, 256])


In [6]:
class DoubleConv(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv = nn.Sequential(

            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(inplace=True),

            nn.Conv2d(
                out_channels,
                out_channels,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)

In [7]:
double_conv = DoubleConv(1, 64)

x = torch.randn(8, 1, 256, 256)

output = double_conv(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)

Input shape: torch.Size([8, 1, 256, 256])
Output shape: torch.Size([8, 64, 256, 256])


In [8]:
class EncoderBlock(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv = DoubleConv(in_channels, out_channels)

        self.pool = nn.MaxPool2d(
            kernel_size=2,
            stride=2
        )

    def forward(self, x):

        features = self.conv(x)

        pooled = self.pool(features)

        return features, pooled

In [9]:
encoder = EncoderBlock(1, 64)

x = torch.randn(8, 1, 256, 256)

features, pooled = encoder(x)

print("Input:", x.shape)
print("Encoder features:", features.shape)
print("Pooled output:", pooled.shape)

Input: torch.Size([8, 1, 256, 256])
Encoder features: torch.Size([8, 64, 256, 256])
Pooled output: torch.Size([8, 64, 128, 128])


In [10]:
class Bottleneck(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x):
        return self.conv(x)

In [11]:
bottleneck = Bottleneck(64, 128)

x = torch.randn(8, 64, 128, 128)

output = bottleneck(x)

print("Input shape:", x.shape)
print("Bottleneck output:", output.shape)

Input shape: torch.Size([8, 64, 128, 128])
Bottleneck output: torch.Size([8, 128, 128, 128])


In [12]:
class DecoderBlock(nn.Module):

    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()

        self.up = nn.ConvTranspose2d(
            in_channels,
            out_channels,
            kernel_size=2,
            stride=2
        )

        self.conv = DoubleConv(
            out_channels + skip_channels,
            out_channels
        )

    def forward(self, x, skip):

        x = self.up(x)

        x = torch.cat([x, skip], dim=1)

        x = self.conv(x)

        return x

In [13]:
decoder = DecoderBlock(
    in_channels=128,
    skip_channels=64,
    out_channels=64
)

x = torch.randn(8, 128, 128, 128)

skip = torch.randn(8, 64, 256, 256)

output = decoder(x, skip)

print("Decoder input:", x.shape)
print("Skip connection:", skip.shape)
print("Decoder output:", output.shape)

Decoder input: torch.Size([8, 128, 128, 128])
Skip connection: torch.Size([8, 64, 256, 256])
Decoder output: torch.Size([8, 64, 256, 256])


In [14]:
class OutputLayer(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=1
        )

    def forward(self, x):
        return self.conv(x)

In [15]:
output_layer = OutputLayer(64, 1)

x = torch.randn(8, 64, 256, 256)

output = output_layer(x)

print("Input shape:", x.shape)
print("Final output shape:", output.shape)

Input shape: torch.Size([8, 64, 256, 256])
Final output shape: torch.Size([8, 1, 256, 256])


In [16]:
class UNet(nn.Module):

    def __init__(self, in_channels=1, out_channels=1):
        super().__init__()

        # Encoder
        self.encoder1 = EncoderBlock(in_channels, 64)
        self.encoder2 = EncoderBlock(64, 128)
        self.encoder3 = EncoderBlock(128, 256)
        self.encoder4 = EncoderBlock(256, 512)

        # Bottleneck
        self.bottleneck = Bottleneck(512, 1024)

        # Decoder
        self.decoder4 = DecoderBlock(1024, 512, 512)
        self.decoder3 = DecoderBlock(512, 256, 256)
        self.decoder2 = DecoderBlock(256, 128, 128)
        self.decoder1 = DecoderBlock(128, 64, 64)

        # Final output
        self.output_layer = OutputLayer(64, out_channels)

    def forward(self, x):

        # Encoder
        skip1, x = self.encoder1(x)
        skip2, x = self.encoder2(x)
        skip3, x = self.encoder3(x)
        skip4, x = self.encoder4(x)

        # Bottleneck
        x = self.bottleneck(x)

        # Decoder with skip connections
        x = self.decoder4(x, skip4)
        x = self.decoder3(x, skip3)
        x = self.decoder2(x, skip2)
        x = self.decoder1(x, skip1)

        # Output
        return self.output_layer(x)

In [17]:
model = UNet(
    in_channels=1,
    out_channels=1
)

print(model)

UNet(
  (encoder1): EncoderBlock(
    (conv): DoubleConv(
      (conv): Sequential(
        (0): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): ReLU(inplace=True)
        (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (3): ReLU(inplace=True)
      )
    )
    (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (encoder2): EncoderBlock(
    (conv): DoubleConv(
      (conv): Sequential(
        (0): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): ReLU(inplace=True)
        (2): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (3): ReLU(inplace=True)
      )
    )
    (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (encoder3): EncoderBlock(
    (conv): DoubleConv(
      (conv): Sequential(
        (0): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): ReLU(inpla

In [18]:
sample_input = torch.randn(2, 1, 256, 256)

with torch.no_grad():
    prediction = model(sample_input)

print("Input shape:", sample_input.shape)
print("Output shape:", prediction.shape)

Input shape: torch.Size([2, 1, 256, 256])
Output shape: torch.Size([2, 1, 256, 256])


In [19]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Total parameters: 31,030,593
Trainable parameters: 31,030,593


## 6. Model Preparation

In this section, we will:
1. Select the available computation device (CPU/GPU).
2. Move the U-Net model to that device.
3. Connect the training and validation DataLoaders.
4. Verify that the model can process an actual microscopy image batch.

In [20]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Selected device:", device)

model = model.to(device)

Selected device: cpu


## 7. Connecting the Dataset

In this section, we will connect the preprocessed microscopy dataset
to our U-Net model using PyTorch DataLoaders.